# Auditoria científica angular dos parâmetros θ para treinamento do Transformer

Este notebook parte do pressuposto de que o banco VQE/COBYLA já foi criado.

Além das análises lineares iniciais, ele verifica:

- periodicidade angular;
- distâncias no toro;
- multimodalidade;
- PCA na representação seno–cosseno;
- dimensão efetiva;
- clusters angulares;
- qualidade energética;
- equivalência por bitstrings e distribuições de saída;
- adequação dos alvos para o Transformer;
- seleção de alvos canônicos;
- validação física opcional por statevector, fidelidade e perturbações locais.

A pergunta científica central é:

> A diversidade observada em θ representa alvos angulares válidos e aprendíveis ou apenas parametrizações redundantes do mesmo comportamento físico?

In [ ]:
from pathlib import Path
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.stats import spearmanr

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=RuntimeWarning)

## 1. Carregar o banco de parâmetros

In [ ]:
RESULTS_DIR = Path("results")
THETA_COLUMN = "best_parameters"

files = sorted(RESULTS_DIR.glob("part_*.pkl"))

df = pd.concat(
    [pd.read_pickle(file) for file in files],
    ignore_index=True,
)

thetas = np.vstack(
    df[THETA_COLUMN].apply(
        lambda values: np.asarray(values, dtype=float).ravel()
    )
)

print("Arquivos carregados:", len(files))
print("Execuções:", thetas.shape[0])
print("Parâmetros por execução:", thetas.shape[1])

## 2. Métricas numéricas de diversidade

In [ ]:
theta_mean = thetas.mean(axis=0)
theta_std = thetas.std(axis=0)
theta_range = np.ptp(thetas, axis=0)

distance_to_first = np.linalg.norm(
    thetas - thetas[0],
    axis=1,
)

distance_to_mean = np.linalg.norm(
    thetas - theta_mean,
    axis=1,
)

unique_12 = np.unique(np.round(thetas, 12), axis=0).shape[0]
unique_8 = np.unique(np.round(thetas, 8), axis=0).shape[0]
unique_6 = np.unique(np.round(thetas, 6), axis=0).shape[0]
unique_4 = np.unique(np.round(thetas, 4), axis=0).shape[0]

identical_to_first = np.all(
    np.isclose(
        thetas,
        thetas[0],
        atol=1e-10,
        rtol=1e-10,
    ),
    axis=1,
)

summary = pd.DataFrame(
    {
        "metrica": [
            "numero_de_execucoes",
            "numero_de_parametros",
            "vetores_unicos_12_casas",
            "vetores_unicos_8_casas",
            "vetores_unicos_6_casas",
            "vetores_unicos_4_casas",
            "desvio_padrao_medio",
            "maior_desvio_padrao",
            "amplitude_media",
            "maior_amplitude",
            "distancia_media_para_primeiro",
            "maior_distancia_para_primeiro",
            "distancia_media_para_media",
            "maior_distancia_para_media",
            "fracao_identica_ao_primeiro",
        ],
        "valor": [
            thetas.shape[0],
            thetas.shape[1],
            unique_12,
            unique_8,
            unique_6,
            unique_4,
            theta_std.mean(),
            theta_std.max(),
            theta_range.mean(),
            theta_range.max(),
            distance_to_first.mean(),
            distance_to_first.max(),
            distance_to_mean.mean(),
            distance_to_mean.max(),
            identical_to_first.mean(),
        ],
    }
)

summary

In [ ]:
parameter_summary = pd.DataFrame(
    {
        "theta": [f"theta_{i}" for i in range(thetas.shape[1])],
        "media": theta_mean,
        "desvio_padrao": theta_std,
        "minimo": thetas.min(axis=0),
        "maximo": thetas.max(axis=0),
        "amplitude": theta_range,
    }
)

parameter_summary

## 3. Mapa de calor dos vetores θ

In [ ]:
plt.figure(figsize=(14, 8))
plt.imshow(
    thetas,
    aspect="auto",
    interpolation="nearest",
)
plt.colorbar(label="Valor de θ")
plt.xlabel("Índice do parâmetro")
plt.ylabel("Execução")
plt.title("Mapa de calor dos parâmetros otimizados")
plt.tight_layout()
plt.show()

## 4. Desvio-padrão de cada parâmetro

In [ ]:
plt.figure(figsize=(13, 6))
plt.bar(
    np.arange(thetas.shape[1]),
    theta_std,
)
plt.xlabel("Índice do parâmetro")
plt.ylabel("Desvio-padrão")
plt.title("Diversidade de cada componente do vetor θ")
plt.tight_layout()
plt.show()

## 5. Amplitude de cada parâmetro

In [ ]:
plt.figure(figsize=(13, 6))
plt.bar(
    np.arange(thetas.shape[1]),
    theta_range,
)
plt.xlabel("Índice do parâmetro")
plt.ylabel("Máximo − mínimo")
plt.title("Amplitude observada em cada componente de θ")
plt.tight_layout()
plt.show()

## 6. Distância para o primeiro vetor

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(
    distance_to_first,
    bins=50,
)
plt.xlabel("Distância euclidiana para o primeiro vetor θ")
plt.ylabel("Número de execuções")
plt.title("Distribuição das distâncias entre soluções")
plt.tight_layout()
plt.show()

## 7. Distância ao vetor médio

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(
    distance_to_mean,
    bins=50,
)
plt.xlabel("Distância euclidiana para o vetor médio")
plt.ylabel("Número de execuções")
plt.title("Dispersão dos parâmetros em torno da média")
plt.tight_layout()
plt.show()

## 8. PCA dos vetores θ

In [ ]:
centered = thetas - theta_mean

_, singular_values, components = np.linalg.svd(
    centered,
    full_matrices=False,
)

pca_coordinates = centered @ components[:2].T

total_variance = np.sum(singular_values ** 2)

if total_variance == 0:
    explained_variance = np.array([0.0, 0.0])
else:
    explained_variance = (
        singular_values[:2] ** 2
    ) / total_variance

plt.figure(figsize=(9, 7))
plt.scatter(
    pca_coordinates[:, 0],
    pca_coordinates[:, 1],
    s=10,
    alpha=0.5,
)
plt.xlabel(
    f"Componente principal 1 ({explained_variance[0] * 100:.4f}%)"
)
plt.ylabel(
    f"Componente principal 2 ({explained_variance[1] * 100:.4f}%)"
)
plt.title("PCA dos parâmetros otimizados")
plt.tight_layout()
plt.show()

## 9. Valores singulares e dimensão efetiva

In [ ]:
normalized_singular_values = singular_values / max(
    singular_values.sum(),
    1e-15,
)

effective_rank = np.exp(
    -np.sum(
        normalized_singular_values
        * np.log(
            np.clip(
                normalized_singular_values,
                1e-15,
                None,
            )
        )
    )
)

print("Posto efetivo:", effective_rank)

plt.figure(figsize=(12, 6))
plt.plot(
    np.arange(1, len(singular_values) + 1),
    singular_values,
    marker="o",
)
plt.xlabel("Componente")
plt.ylabel("Valor singular")
plt.title("Espectro de diversidade dos vetores θ")
plt.tight_layout()
plt.show()

## 10. Correlação entre os parâmetros

In [ ]:
theta_correlation = np.corrcoef(
    thetas,
    rowvar=False,
)

plt.figure(figsize=(10, 9))
plt.imshow(
    theta_correlation,
    aspect="auto",
    interpolation="nearest",
    vmin=-1,
    vmax=1,
)
plt.colorbar(label="Correlação")
plt.xlabel("Índice do parâmetro")
plt.ylabel("Índice do parâmetro")
plt.title("Correlação entre componentes de θ")
plt.tight_layout()
plt.show()

## 11. Salvar tabelas de análise

In [ ]:
ANALYSIS_DIR = RESULTS_DIR / "theta_scientific_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(
    ANALYSIS_DIR / "theta_diversity_summary.csv",
    index=False,
)

parameter_summary.to_csv(
    ANALYSIS_DIR / "theta_parameter_summary.csv",
    index=False,
)

print("Arquivos salvos em:", ANALYSIS_DIR.resolve())

## Interpretação

Ausência de diversidade paramétrica é indicada por:

- um único vetor após arredondamento;
- desvios-padrão e amplitudes próximos de zero;
- distâncias concentradas em zero;
- todos os pontos sobrepostos no PCA;
- apenas um valor singular relevante ou todos iguais a zero.

Pequenas diferenças apenas nas últimas casas decimais representam variação numérica, não diversidade estrutural.

# Parte II — Análises angulares e validade para aprendizado

As métricas anteriores tratam θ como vetor euclidiano. A partir daqui, cada componente é tratada como variável periódica.

## 12. Configuração das análises avançadas

In [ ]:
ANALYSIS_DIR = RESULTS_DIR / "theta_scientific_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

ENERGY_COLUMN = "objective_function_value"
EXACT_ENERGY_COLUMN = "best_objective_function_value"
PROBABILITY_COLUMN = "probability_best_answer"
COUNTS_COLUMN = "counts"
BITSTRING_COLUMN = "most_frequent_bitstring"
CALLBACK_COLUMN = "callback_list"

RANDOM_SEED = 42
PAIR_SAMPLE_SIZE = 30_000
SILHOUETTE_SAMPLE_SIZE = 5_000
K_MIN = 2
K_MAX = 8
TOP_ANGULAR_PLOTS = 8
ENERGY_EQ_TOL = 1e-3

rng = np.random.default_rng(RANDOM_SEED)
n_runs, theta_dim = thetas.shape

## 13. Representação angular

Usamos:

\[
\operatorname{wrap}(\theta)=\operatorname{atan2}(\sin\theta,\cos\theta)
\]

para levar os ângulos a \([-\pi,\pi)\), e o embedding:

\[
z(\theta)=(\cos\theta_1,\sin\theta_1,\ldots,\cos\theta_d,\sin\theta_d).
\]

In [ ]:
def wrap_angles(values):
    values = np.asarray(values, dtype=float)
    return np.arctan2(np.sin(values), np.cos(values))


def angular_delta(theta_a, theta_b):
    return wrap_angles(np.asarray(theta_a) - np.asarray(theta_b))


def toroidal_distance(theta_a, theta_b):
    return float(np.linalg.norm(angular_delta(theta_a, theta_b)))


thetas_wrapped = wrap_angles(thetas)
theta_embedding = np.concatenate(
    [np.cos(thetas_wrapped), np.sin(thetas_wrapped)],
    axis=1,
)

print("Theta original:", thetas.shape)
print("Embedding seno-cosseno:", theta_embedding.shape)

## 14. Estatística circular por parâmetro

In [ ]:
cos_mean = np.mean(np.cos(thetas_wrapped), axis=0)
sin_mean = np.mean(np.sin(thetas_wrapped), axis=0)

circular_mean = np.arctan2(sin_mean, cos_mean)
resultant_length = np.sqrt(cos_mean**2 + sin_mean**2)
circular_variance = 1.0 - resultant_length
circular_std = np.sqrt(
    -2.0 * np.log(np.clip(resultant_length, 1e-15, 1.0))
)

circular_stats = pd.DataFrame({
    "theta_index": np.arange(theta_dim),
    "circular_mean": circular_mean,
    "resultant_length_R": resultant_length,
    "circular_variance": circular_variance,
    "circular_std": circular_std,
    "linear_std_raw": np.std(thetas, axis=0),
    "linear_std_wrapped": np.std(thetas_wrapped, axis=0),
})

circular_stats.to_csv(
    ANALYSIS_DIR / "01_circular_statistics.csv",
    index=False,
)

circular_stats

In [ ]:
plt.figure(figsize=(13, 6))
plt.bar(np.arange(theta_dim), circular_variance)
plt.xlabel("Índice do parâmetro")
plt.ylabel("Variância circular")
plt.title("Variância circular de cada componente de θ")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "01_circular_variance.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(13, 6))
plt.bar(np.arange(theta_dim), resultant_length)
plt.xlabel("Índice do parâmetro")
plt.ylabel("Comprimento resultante R")
plt.title("Concentração angular de cada componente de θ")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "02_resultant_length.png", dpi=300)
plt.show()

Interpretação:

- \(R\approx1\): forte concentração angular;
- \(R\approx0\): distribuição espalhada, uniforme ou multimodal;
- desvio linear alto e variância circular baixa: possível efeito artificial da fronteira \(0/2\pi\).

## 15. Diagnóstico de multimodalidade angular

In [ ]:
def circular_mode_count(
    angles,
    bins=72,
    smooth_sigma=1.5,
    prominence_fraction=0.08,
):
    hist, edges = np.histogram(
        wrap_angles(angles),
        bins=bins,
        range=(-np.pi, np.pi),
        density=True,
    )
    smooth = gaussian_filter1d(
        hist.astype(float),
        sigma=smooth_sigma,
        mode="wrap",
    )
    prominence = max(float(np.max(smooth)) * prominence_fraction, 1e-12)
    peaks, _ = find_peaks(smooth, prominence=prominence)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return {
        "mode_count": int(len(peaks)),
        "peak_angles": centers[peaks],
        "histogram": hist,
        "smooth_density": smooth,
        "centers": centers,
    }


mode_rows = []
for index in range(theta_dim):
    result = circular_mode_count(thetas_wrapped[:, index])
    mode_rows.append({
        "theta_index": index,
        "estimated_mode_count": result["mode_count"],
        "peak_angles": result["peak_angles"].tolist(),
        "circular_variance": circular_variance[index],
        "resultant_length_R": resultant_length[index],
    })

mode_table = pd.DataFrame(mode_rows)
mode_table.to_csv(
    ANALYSIS_DIR / "02_angular_mode_diagnostics.csv",
    index=False,
)
mode_table

In [ ]:
plt.figure(figsize=(13, 6))
plt.bar(mode_table["theta_index"], mode_table["estimated_mode_count"])
plt.xlabel("Índice do parâmetro")
plt.ylabel("Número estimado de modos")
plt.title("Diagnóstico de multimodalidade angular")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "03_angular_mode_count.png", dpi=300)
plt.show()

In [ ]:
plot_indices = (
    mode_table.sort_values(
        ["estimated_mode_count", "circular_variance"],
        ascending=False,
    )
    .head(TOP_ANGULAR_PLOTS)["theta_index"]
    .tolist()
)

for index in plot_indices:
    result = circular_mode_count(thetas_wrapped[:, index])
    plt.figure(figsize=(10, 5))
    plt.hist(
        thetas_wrapped[:, index],
        bins=72,
        range=(-np.pi, np.pi),
        density=True,
        alpha=0.45,
    )
    plt.plot(result["centers"], result["smooth_density"], linewidth=2)
    plt.xlabel(f"θ{index} em [-π, π)")
    plt.ylabel("Densidade")
    plt.title(
        f"Distribuição angular de θ{index} "
        f"({result['mode_count']} modos estimados)"
    )
    plt.tight_layout()
    plt.savefig(
        ANALYSIS_DIR / f"angular_distribution_theta_{index:02d}.png",
        dpi=300,
    )
    plt.show()

O número de modos é um diagnóstico de densidade suavizada. Ele deve ser interpretado junto com energia, bitstrings e clusters.

## 16. Distâncias toroidais

In [ ]:
distance_to_first_torus = np.linalg.norm(
    angular_delta(thetas_wrapped, thetas_wrapped[0]),
    axis=1,
)

distance_to_circular_mean = np.linalg.norm(
    angular_delta(thetas_wrapped, circular_mean),
    axis=1,
)

print("Distância média ao primeiro:", distance_to_first_torus.mean())
print("Distância média à média circular:", distance_to_circular_mean.mean())

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(distance_to_first_torus, bins=50)
plt.xlabel("Distância toroidal para o primeiro vetor")
plt.ylabel("Número de execuções")
plt.title("Distâncias angulares para o primeiro vetor θ")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "04_toroidal_distance_to_first.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(distance_to_circular_mean, bins=50)
plt.xlabel("Distância toroidal para a média circular")
plt.ylabel("Número de execuções")
plt.title("Dispersão angular em torno da média circular")
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "05_toroidal_distance_to_circular_mean.png",
    dpi=300,
)
plt.show()

### Distâncias entre pares e vizinhos mais próximos

In [ ]:
pair_count = min(PAIR_SAMPLE_SIZE, max(n_runs * 3, 1))
pair_i = rng.integers(0, n_runs, size=pair_count)
pair_j = rng.integers(0, n_runs, size=pair_count)
mask = pair_i != pair_j
pair_i = pair_i[mask]
pair_j = pair_j[mask]

pairwise_toroidal_distance = np.linalg.norm(
    angular_delta(thetas_wrapped[pair_i], thetas_wrapped[pair_j]),
    axis=1,
)

plt.figure(figsize=(12, 6))
plt.hist(pairwise_toroidal_distance, bins=60)
plt.xlabel("Distância toroidal entre pares")
plt.ylabel("Número de pares")
plt.title("Distribuição amostrada das distâncias entre soluções")
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "06_sampled_pairwise_toroidal_distance.png",
    dpi=300,
)
plt.show()

In [ ]:
neighbors = NearestNeighbors(n_neighbors=2, metric="euclidean")
neighbors.fit(theta_embedding)
_, neighbor_indices = neighbors.kneighbors(theta_embedding)
nearest_index = neighbor_indices[:, 1]

nearest_toroidal_distance = np.linalg.norm(
    angular_delta(thetas_wrapped, thetas_wrapped[nearest_index]),
    axis=1,
)

plt.figure(figsize=(12, 6))
plt.hist(nearest_toroidal_distance, bins=60)
plt.xlabel("Distância toroidal para o vizinho mais próximo")
plt.ylabel("Número de execuções")
plt.title("Redundância local no banco de θ")
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "07_nearest_neighbor_toroidal_distance.png",
    dpi=300,
)
plt.show()

## 17. PCA angular no embedding seno-cosseno

In [ ]:
pca_angular = PCA(random_state=RANDOM_SEED)
pca_angular_coordinates = pca_angular.fit_transform(theta_embedding)

explained_ratio = pca_angular.explained_variance_ratio_
cumulative_ratio = np.cumsum(explained_ratio)
components_90 = int(np.searchsorted(cumulative_ratio, 0.90) + 1)
components_95 = int(np.searchsorted(cumulative_ratio, 0.95) + 1)

print("PC1:", explained_ratio[0])
print("PC2:", explained_ratio[1])
print("Componentes para 90%:", components_90)
print("Componentes para 95%:", components_95)

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(
    pca_angular_coordinates[:, 0],
    pca_angular_coordinates[:, 1],
    s=9,
    alpha=0.45,
)
plt.xlabel(f"PC1 ({explained_ratio[0] * 100:.3f}%)")
plt.ylabel(f"PC2 ({explained_ratio[1] * 100:.3f}%)")
plt.title("PCA angular no embedding seno-cosseno")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "08_pca_sincos.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
    np.arange(1, len(cumulative_ratio) + 1),
    cumulative_ratio,
    marker="o",
    markersize=3,
)
plt.axhline(0.90, linestyle="--")
plt.axhline(0.95, linestyle="--")
plt.xlabel("Número de componentes")
plt.ylabel("Variância explicada acumulada")
plt.title("Dimensão necessária para representar a diversidade angular")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "09_pca_cumulative_variance.png", dpi=300)
plt.show()

## 18. Dimensão efetiva

In [ ]:
eigenvalues = np.asarray(pca_angular.explained_variance_, dtype=float)
p_eigen = eigenvalues / max(eigenvalues.sum(), 1e-15)

effective_rank = float(
    np.exp(
        -np.sum(p_eigen * np.log(np.clip(p_eigen, 1e-15, None)))
    )
)

participation_ratio = float(
    (eigenvalues.sum() ** 2)
    / max(np.sum(eigenvalues**2), 1e-15)
)

print("Posto efetivo entrópico:", effective_rank)
print("Participation ratio:", participation_ratio)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(np.arange(1, len(eigenvalues) + 1), eigenvalues, marker="o")
plt.xlabel("Componente")
plt.ylabel("Autovalor")
plt.title("Espectro da diversidade angular")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "10_angular_eigenspectrum.png", dpi=300)
plt.show()

## 19. Clustering angular

In [ ]:
sample_size = min(SILHOUETTE_SAMPLE_SIZE, n_runs)
sample_indices = rng.choice(n_runs, size=sample_size, replace=False)
sample_embedding = theta_embedding[sample_indices]

silhouette_rows = []
for k in range(K_MIN, K_MAX + 1):
    model = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20)
    labels_sample = model.fit_predict(sample_embedding)
    silhouette_rows.append({
        "k": k,
        "silhouette_score": float(
            silhouette_score(sample_embedding, labels_sample)
        ),
    })

silhouette_table = pd.DataFrame(silhouette_rows)
best_k = int(
    silhouette_table.loc[
        silhouette_table["silhouette_score"].idxmax(),
        "k",
    ]
)

print("Melhor número de clusters:", best_k)
silhouette_table

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    silhouette_table["k"],
    silhouette_table["silhouette_score"],
    marker="o",
)
plt.xlabel("Número de clusters")
plt.ylabel("Silhouette score")
plt.title("Seleção do número de clusters angulares")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "11_cluster_silhouette.png", dpi=300)
plt.show()

In [ ]:
cluster_model = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_SEED,
    n_init=30,
)
cluster_labels = cluster_model.fit_predict(theta_embedding)
df = df.copy()
df["theta_cluster"] = cluster_labels

plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    pca_angular_coordinates[:, 0],
    pca_angular_coordinates[:, 1],
    c=cluster_labels,
    s=9,
    alpha=0.5,
)
plt.xlabel("PC1 angular")
plt.ylabel("PC2 angular")
plt.title("Clusters no espaço angular")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "12_pca_angular_clusters.png", dpi=300)
plt.show()

### Medoides e resumo dos clusters

In [ ]:
cluster_medoid_rows = []
cluster_medoid_indices = []

for cluster_id in range(best_k):
    members = np.where(cluster_labels == cluster_id)[0]
    member_embedding = theta_embedding[members]
    center = cluster_model.cluster_centers_[cluster_id]
    local_index = np.argmin(np.linalg.norm(member_embedding - center, axis=1))
    global_index = int(members[local_index])
    cluster_medoid_indices.append(global_index)

    row = {
        "cluster": cluster_id,
        "size": int(len(members)),
        "medoid_row_index": global_index,
        "mean_toroidal_distance_to_medoid": float(
            np.mean(
                np.linalg.norm(
                    angular_delta(
                        thetas_wrapped[members],
                        thetas_wrapped[global_index],
                    ),
                    axis=1,
                )
            )
        ),
    }

    if ENERGY_COLUMN in df.columns:
        values = pd.to_numeric(df.loc[members, ENERGY_COLUMN], errors="coerce")
        row["mean_energy"] = float(values.mean())
        row["min_energy"] = float(values.min())

    if PROBABILITY_COLUMN in df.columns:
        values = pd.to_numeric(
            df.loc[members, PROBABILITY_COLUMN],
            errors="coerce",
        )
        row["mean_probability_best_answer"] = float(values.mean())

    if BITSTRING_COLUMN in df.columns:
        bitstrings = df.loc[members, BITSTRING_COLUMN].astype(str)
        row["dominant_bitstring"] = bitstrings.value_counts().index[0]
        row["dominant_bitstring_fraction"] = float(
            bitstrings.value_counts(normalize=True).iloc[0]
        )

    cluster_medoid_rows.append(row)

cluster_summary = pd.DataFrame(cluster_medoid_rows)
cluster_summary.to_csv(
    ANALYSIS_DIR / "03_cluster_summary.csv",
    index=False,
)
cluster_summary

## 20. Qualidade energética

In [ ]:
energy_available = ENERGY_COLUMN in df.columns
exact_available = EXACT_ENERGY_COLUMN in df.columns

if energy_available:
    energy = pd.to_numeric(df[ENERGY_COLUMN], errors="coerce").to_numpy(float)
else:
    energy = np.full(n_runs, np.nan)

if exact_available:
    exact_energy = pd.to_numeric(
        df[EXACT_ENERGY_COLUMN],
        errors="coerce",
    ).to_numpy(float)
    signed_gap = energy - exact_energy
    absolute_gap = np.abs(signed_gap)
    relative_gap = absolute_gap / np.maximum(np.abs(exact_energy), 1e-12)
else:
    exact_energy = np.full(n_runs, np.nan)
    signed_gap = np.full(n_runs, np.nan)
    absolute_gap = np.full(n_runs, np.nan)
    relative_gap = np.full(n_runs, np.nan)

quality_table = pd.DataFrame({
    "row_index": np.arange(n_runs),
    "theta_cluster": cluster_labels,
    "energy": energy,
    "exact_energy": exact_energy,
    "signed_gap": signed_gap,
    "absolute_gap": absolute_gap,
    "relative_gap": relative_gap,
    "distance_to_circular_mean": distance_to_circular_mean,
    "nearest_neighbor_distance": nearest_toroidal_distance,
})
quality_table.to_csv(
    ANALYSIS_DIR / "04_energy_quality_by_theta.csv",
    index=False,
)
quality_table.describe()

In [ ]:
if np.isfinite(absolute_gap).any():
    finite_gap = absolute_gap[np.isfinite(absolute_gap)]
    plt.figure(figsize=(12, 6))
    plt.hist(finite_gap, bins=60)
    plt.xlabel("|Energia VQE − energia de referência|")
    plt.ylabel("Número de execuções")
    plt.title("Distribuição do erro energético")
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "13_energy_absolute_gap.png", dpi=300)
    plt.show()

In [ ]:
if np.isfinite(absolute_gap).any():
    valid = np.isfinite(absolute_gap) & np.isfinite(distance_to_circular_mean)
    plt.figure(figsize=(10, 7))
    plt.scatter(
        distance_to_circular_mean[valid],
        absolute_gap[valid],
        s=9,
        alpha=0.35,
    )
    plt.xlabel("Distância toroidal para a média circular")
    plt.ylabel("Erro absoluto de energia")
    plt.title("Diversidade angular versus qualidade energética")
    plt.tight_layout()
    plt.savefig(
        ANALYSIS_DIR / "14_angular_distance_vs_energy_gap.png",
        dpi=300,
    )
    plt.show()
    print(
        "Spearman:",
        spearmanr(distance_to_circular_mean[valid], absolute_gap[valid]),
    )

## 21. Distribuições de bitstrings e divergência Jensen–Shannon

In [ ]:
def normalize_counts(counts):
    if not isinstance(counts, dict) or not counts:
        return {}
    total = float(sum(counts.values()))
    if total <= 0:
        return {}
    return {str(key): float(value) / total for key, value in counts.items()}


def js_divergence_dict(prob_a, prob_b):
    keys = sorted(set(prob_a) | set(prob_b))
    if not keys:
        return np.nan
    p = np.array([prob_a.get(key, 0.0) for key in keys])
    q = np.array([prob_b.get(key, 0.0) for key in keys])
    p = p / max(p.sum(), 1e-15)
    q = q / max(q.sum(), 1e-15)
    m = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return np.sum(
            a[mask] * np.log2(a[mask] / np.clip(b[mask], 1e-15, None))
        )

    return float(
        0.5 * kl_divergence(p, m)
        + 0.5 * kl_divergence(q, m)
    )

In [ ]:
if COUNTS_COLUMN in df.columns:
    normalized_counts = [normalize_counts(value) for value in df[COUNTS_COLUMN]]
    js_values = np.empty(len(pair_i), dtype=float)

    for position, (index_a, index_b) in enumerate(zip(pair_i, pair_j)):
        js_values[position] = js_divergence_dict(
            normalized_counts[index_a],
            normalized_counts[index_b],
        )

    valid = np.isfinite(js_values) & np.isfinite(pairwise_toroidal_distance)
    plt.figure(figsize=(10, 7))
    plt.scatter(
        pairwise_toroidal_distance[valid],
        js_values[valid],
        s=8,
        alpha=0.25,
    )
    plt.xlabel("Distância toroidal entre θ")
    plt.ylabel("Divergência Jensen–Shannon")
    plt.title("Diferença paramétrica versus diferença de saída")
    plt.tight_layout()
    plt.savefig(
        ANALYSIS_DIR / "15_toroidal_distance_vs_js_divergence.png",
        dpi=300,
    )
    plt.show()
    print(
        "Spearman distância angular × JS:",
        spearmanr(pairwise_toroidal_distance[valid], js_values[valid]),
    )

- distância angular alta e JS baixa: redundância paramétrica;
- distância angular alta e JS alta: diversidade física de saída;
- distância angular baixa e JS alta: alta sensibilidade local.

## 22. Classes aproximadas de equivalência física

In [ ]:
equivalence_table = pd.DataFrame({
    "row_index": np.arange(n_runs),
    "theta_cluster": cluster_labels,
})

equivalence_table["dominant_bitstring"] = (
    df[BITSTRING_COLUMN].astype(str).to_numpy()
    if BITSTRING_COLUMN in df.columns
    else "unknown"
)

equivalence_table["energy_bin"] = (
    np.round(energy / ENERGY_EQ_TOL).astype("Int64")
    if np.isfinite(energy).any()
    else -1
)

equivalence_table["equivalence_class"] = (
    equivalence_table["dominant_bitstring"].astype(str)
    + "__"
    + equivalence_table["energy_bin"].astype(str)
)

equivalence_counts = (
    equivalence_table["equivalence_class"]
    .value_counts()
    .rename_axis("equivalence_class")
    .reset_index(name="size")
)

equivalence_counts.to_csv(
    ANALYSIS_DIR / "05_equivalence_class_counts.csv",
    index=False,
)

print("Classes aproximadas:", len(equivalence_counts))
print("Maior classe:", int(equivalence_counts["size"].max()))
equivalence_counts.head(20)

In [ ]:
class_spread_rows = []
for class_name, indices in equivalence_table.groupby(
    "equivalence_class"
).groups.items():
    indices = np.asarray(list(indices), dtype=int)
    if len(indices) < 2:
        continue
    reference = indices[0]
    spread = np.linalg.norm(
        angular_delta(
            thetas_wrapped[indices],
            thetas_wrapped[reference],
        ),
        axis=1,
    )
    class_spread_rows.append({
        "equivalence_class": class_name,
        "size": int(len(indices)),
        "mean_angular_spread": float(spread.mean()),
        "max_angular_spread": float(spread.max()),
    })

equivalence_spread = pd.DataFrame(class_spread_rows)
equivalence_spread.to_csv(
    ANALYSIS_DIR / "06_equivalence_class_angular_spread.csv",
    index=False,
)
equivalence_spread.sort_values("size", ascending=False).head(20)

## 23. Auditoria do ponto inicial e trajetória do COBYLA

In [ ]:
if "initial_point" in df.columns:
    initial_points = np.vstack(
        df["initial_point"].apply(
            lambda values: np.asarray(values, dtype=float).ravel()
        )
    )
    unique_initial_points = np.unique(
        np.round(initial_points, 10),
        axis=0,
    ).shape[0]
    displacement_from_initial = np.linalg.norm(
        angular_delta(thetas_wrapped, initial_points),
        axis=1,
    )
    print("Pontos iniciais únicos:", unique_initial_points)
    print("Deslocamento angular médio:", displacement_from_initial.mean())

    plt.figure(figsize=(12, 6))
    plt.hist(displacement_from_initial, bins=60)
    plt.xlabel("Distância toroidal entre θ inicial e θ final")
    plt.ylabel("Número de execuções")
    plt.title("Quanto o COBYLA deslocou os parâmetros")
    plt.tight_layout()
    plt.savefig(
        ANALYSIS_DIR / "16_displacement_from_initial_point.png",
        dpi=300,
    )
    plt.show()

In [ ]:
trajectory_rows = []
if CALLBACK_COLUMN in df.columns:
    sample_indices_trajectory = rng.choice(
        n_runs,
        size=min(100, n_runs),
        replace=False,
    )
    for row_index in sample_indices_trajectory:
        callback = df.iloc[row_index][CALLBACK_COLUMN]
        if not isinstance(callback, (list, tuple)) or len(callback) == 0:
            continue
        callback_energies = []
        for item in callback:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                callback_energies.append(float(np.real(item[1])))
        if len(callback_energies) < 2:
            continue
        trajectory_rows.append({
            "row_index": int(row_index),
            "evaluations": len(callback_energies),
            "initial_energy": callback_energies[0],
            "final_energy": callback_energies[-1],
            "best_energy": min(callback_energies),
            "energy_improvement": callback_energies[0] - min(callback_energies),
        })

trajectory_summary = pd.DataFrame(trajectory_rows)
if not trajectory_summary.empty:
    trajectory_summary.to_csv(
        ANALYSIS_DIR / "07_cobyla_trajectory_summary.csv",
        index=False,
    )
trajectory_summary.describe() if not trajectory_summary.empty else trajectory_summary

## 24. Adequação dos alvos para o Transformer

In [ ]:
linear_mean_target = np.mean(thetas, axis=0)
circular_mean_target = circular_mean.copy()

distance_to_linear_mean_target = np.linalg.norm(
    angular_delta(thetas_wrapped, linear_mean_target),
    axis=1,
)
distance_to_circular_target = np.linalg.norm(
    angular_delta(thetas_wrapped, circular_mean_target),
    axis=1,
)

global_medoid_index = int(
    np.argmin(
        np.linalg.norm(
            theta_embedding - theta_embedding.mean(axis=0),
            axis=1,
        )
    )
)
global_medoid = thetas_wrapped[global_medoid_index]
distance_to_global_medoid = np.linalg.norm(
    angular_delta(thetas_wrapped, global_medoid),
    axis=1,
)

representative_summary = pd.DataFrame({
    "representative": ["linear_mean", "circular_mean", "global_medoid"],
    "mean_toroidal_distance": [
        distance_to_linear_mean_target.mean(),
        distance_to_circular_target.mean(),
        distance_to_global_medoid.mean(),
    ],
    "max_toroidal_distance": [
        distance_to_linear_mean_target.max(),
        distance_to_circular_target.max(),
        distance_to_global_medoid.max(),
    ],
})
representative_summary.to_csv(
    ANALYSIS_DIR / "08_representative_target_comparison.csv",
    index=False,
)
representative_summary

In [ ]:
multimodal_fraction = float(
    np.mean(mode_table["estimated_mode_count"] > 1)
)

transformer_audit = pd.DataFrame({
    "metric": [
        "number_of_runs",
        "theta_dimension",
        "mean_circular_variance",
        "max_circular_variance",
        "fraction_multimodal_parameters",
        "effective_rank",
        "participation_ratio",
        "pca_components_90_percent",
        "pca_components_95_percent",
        "mean_nearest_neighbor_toroidal_distance",
        "best_cluster_count",
        "global_medoid_index",
    ],
    "value": [
        n_runs,
        theta_dim,
        circular_variance.mean(),
        circular_variance.max(),
        multimodal_fraction,
        effective_rank,
        participation_ratio,
        components_90,
        components_95,
        nearest_toroidal_distance.mean(),
        best_k,
        global_medoid_index,
    ],
})
transformer_audit.to_csv(
    ANALYSIS_DIR / "09_transformer_target_audit.csv",
    index=False,
)
transformer_audit

Uma regressão por MSE linear é inadequada quando a distribuição é multimodal ou quando a média linear fica distante das soluções reais.

Representações recomendadas:

\[
(\cos\theta_i,\sin\theta_i)
\]

ou perda periódica:

\[
\mathcal L_{\mathrm{circ}}=
\frac1d\sum_i\left[1-\cos(\hat\theta_i-\theta_i)\right].
\]

Para multimodalidade, use cluster + regressão residual, múltiplos candidatos, mixture density head ou alvo canônico.

## 25. Exportar banco angular pronto para treinamento

In [ ]:
training_columns = {}
for index in range(theta_dim):
    training_columns[f"theta_{index:02d}"] = thetas_wrapped[:, index]
    training_columns[f"theta_sin_{index:02d}"] = np.sin(
        thetas_wrapped[:, index]
    )
    training_columns[f"theta_cos_{index:02d}"] = np.cos(
        thetas_wrapped[:, index]
    )

training_bank = pd.DataFrame(training_columns)
training_bank["theta_cluster"] = cluster_labels
training_bank["nearest_neighbor_toroidal_distance"] = (
    nearest_toroidal_distance
)

if energy_available:
    training_bank["energy"] = energy
if exact_available:
    training_bank["absolute_energy_gap"] = absolute_gap
    training_bank["relative_energy_gap"] = relative_gap
if PROBABILITY_COLUMN in df.columns:
    training_bank["probability_best_answer"] = pd.to_numeric(
        df[PROBABILITY_COLUMN],
        errors="coerce",
    ).to_numpy()

training_bank.to_pickle(
    ANALYSIS_DIR / "theta_transformer_training_bank.pkl"
)
np.save(ANALYSIS_DIR / "theta_wrapped.npy", thetas_wrapped)
np.save(ANALYSIS_DIR / "theta_sincos_embedding.npy", theta_embedding)

print("Banco angular exportado:", training_bank.shape)

### Alvos canônicos por cluster

In [ ]:
canonical_rows = []
for cluster_id, medoid_index in enumerate(cluster_medoid_indices):
    row = {
        "cluster": cluster_id,
        "medoid_row_index": medoid_index,
        "theta_medoid": thetas_wrapped[medoid_index],
        "theta_sincos_medoid": theta_embedding[medoid_index],
    }
    if energy_available:
        row["energy"] = energy[medoid_index]
    if exact_available:
        row["absolute_energy_gap"] = absolute_gap[medoid_index]
    if PROBABILITY_COLUMN in df.columns:
        row["probability_best_answer"] = pd.to_numeric(
            pd.Series([df.iloc[medoid_index][PROBABILITY_COLUMN]]),
            errors="coerce",
        ).iloc[0]
    if BITSTRING_COLUMN in df.columns:
        row["dominant_bitstring"] = str(
            df.iloc[medoid_index][BITSTRING_COLUMN]
        )
    canonical_rows.append(row)

canonical_targets = pd.DataFrame(canonical_rows)
canonical_targets.to_pickle(
    ANALYSIS_DIR / "theta_canonical_cluster_medoids.pkl"
)
canonical_targets[
    [
        column
        for column in canonical_targets.columns
        if column not in {"theta_medoid", "theta_sincos_medoid"}
    ]
]

## 26. Validação física opcional por Qiskit

Para reavaliar energias exatamente, calcular fidelidades e testar mínimos locais, salve no notebook de geração:

```python
import pickle
with open("results/problem_bundle.pkl", "wb") as file:
    pickle.dump(
        {"ansatz_base": ansatz_base, "ising": ising, "offset": offset},
        file,
    )
```

Depois ative `RUN_QISKIT_VALIDATION`.

In [ ]:
RUN_QISKIT_VALIDATION = False
PROBLEM_BUNDLE_PATH = RESULTS_DIR / "problem_bundle.pkl"
QISKIT_VALIDATION_SAMPLE = min(1_000, n_runs)
LOCAL_TEST_PERTURBATIONS = 100
LOCAL_TEST_SCALE = 1e-2

In [ ]:
if RUN_QISKIT_VALIDATION:
    from qiskit.primitives import StatevectorEstimator
    from qiskit.quantum_info import Statevector, state_fidelity

    with PROBLEM_BUNDLE_PATH.open("rb") as file:
        bundle = pickle.load(file)

    ansatz_base = bundle["ansatz_base"]
    ising = bundle["ising"]
    offset = float(bundle["offset"])
    exact_estimator = StatevectorEstimator()

    def exact_statevector_energy(theta):
        result = exact_estimator.run([
            (ansatz_base, ising, np.asarray(theta, dtype=float))
        ]).result()
        return float(np.real(result[0].data.evs)) + offset

    def statevector_for_theta(theta):
        circuit = ansatz_base.assign_parameters(
            np.asarray(theta, dtype=float),
            inplace=False,
        )
        return Statevector.from_instruction(circuit)

    validation_indices = rng.choice(
        n_runs,
        size=QISKIT_VALIDATION_SAMPLE,
        replace=False,
    )
    exact_revalidated_energy = np.array([
        exact_statevector_energy(thetas[index])
        for index in validation_indices
    ])

    exact_validation_table = pd.DataFrame({
        "row_index": validation_indices,
        "exact_statevector_energy": exact_revalidated_energy,
        "stored_energy": energy[validation_indices],
        "theta_cluster": cluster_labels[validation_indices],
    })
    exact_validation_table["stored_minus_exact"] = (
        exact_validation_table["stored_energy"]
        - exact_validation_table["exact_statevector_energy"]
    )
    exact_validation_table.to_csv(
        ANALYSIS_DIR / "10_exact_statevector_revalidation.csv",
        index=False,
    )
    exact_validation_table.describe()

### Fidelidade entre medoides

In [ ]:
if RUN_QISKIT_VALIDATION:
    medoid_states = [
        statevector_for_theta(thetas[index])
        for index in cluster_medoid_indices
    ]
    fidelity_matrix = np.zeros((best_k, best_k), dtype=float)
    for index_a in range(best_k):
        for index_b in range(best_k):
            fidelity_matrix[index_a, index_b] = state_fidelity(
                medoid_states[index_a],
                medoid_states[index_b],
            )

    pd.DataFrame(fidelity_matrix).to_csv(
        ANALYSIS_DIR / "11_cluster_medoid_fidelity.csv",
        index=False,
    )

    plt.figure(figsize=(8, 7))
    plt.imshow(fidelity_matrix, vmin=0, vmax=1, interpolation="nearest")
    plt.colorbar(label="Fidelidade")
    plt.xlabel("Cluster")
    plt.ylabel("Cluster")
    plt.title("Fidelidade entre estados dos medoides")
    plt.tight_layout()
    plt.savefig(
        ANALYSIS_DIR / "17_cluster_medoid_fidelity.png",
        dpi=300,
    )
    plt.show()

### Perturbações locais

In [ ]:
if RUN_QISKIT_VALIDATION:
    local_minimum_rows = []
    for cluster_id, medoid_index in enumerate(cluster_medoid_indices):
        theta_reference = thetas[medoid_index]
        reference_energy = exact_statevector_energy(theta_reference)
        perturbations = rng.normal(
            0.0,
            LOCAL_TEST_SCALE,
            size=(LOCAL_TEST_PERTURBATIONS, theta_dim),
        )
        perturbed_energies = np.array([
            exact_statevector_energy(
                wrap_angles(theta_reference + perturbation)
            )
            for perturbation in perturbations
        ])
        local_minimum_rows.append({
            "cluster": cluster_id,
            "medoid_row_index": medoid_index,
            "reference_energy": reference_energy,
            "fraction_perturbations_with_higher_energy": float(
                np.mean(perturbed_energies >= reference_energy)
            ),
            "mean_energy_increase": float(
                np.mean(perturbed_energies - reference_energy)
            ),
            "minimum_energy_change": float(
                np.min(perturbed_energies - reference_energy)
            ),
        })

    local_minimum_table = pd.DataFrame(local_minimum_rows)
    local_minimum_table.to_csv(
        ANALYSIS_DIR / "12_local_minimum_perturbation_test.csv",
        index=False,
    )
    local_minimum_table

## 27. Resumo científico automático

In [ ]:
summary_scientific = {
    "number_of_runs": int(n_runs),
    "theta_dimension": int(theta_dim),
    "unique_wrapped_vectors_6_decimals": int(
        np.unique(np.round(thetas_wrapped, 6), axis=0).shape[0]
    ),
    "mean_circular_variance": float(circular_variance.mean()),
    "max_circular_variance": float(circular_variance.max()),
    "fraction_multimodal_parameters": float(multimodal_fraction),
    "mean_toroidal_distance_to_circular_mean": float(
        distance_to_circular_mean.mean()
    ),
    "mean_nearest_neighbor_toroidal_distance": float(
        nearest_toroidal_distance.mean()
    ),
    "effective_rank": float(effective_rank),
    "participation_ratio": float(participation_ratio),
    "pca_components_90_percent": int(components_90),
    "pca_components_95_percent": int(components_95),
    "best_cluster_count": int(best_k),
    "number_of_equivalence_classes": int(len(equivalence_counts)),
    "global_medoid_index": int(global_medoid_index),
}

if np.isfinite(absolute_gap).any():
    summary_scientific["median_absolute_energy_gap"] = float(
        np.nanmedian(absolute_gap)
    )
    summary_scientific["fraction_gap_below_1e_minus_2"] = float(
        np.nanmean(absolute_gap <= 1e-2)
    )
    summary_scientific["fraction_gap_below_1e_minus_3"] = float(
        np.nanmean(absolute_gap <= 1e-3)
    )

(ANALYSIS_DIR / "scientific_summary.json").write_text(
    json.dumps(summary_scientific, indent=2),
    encoding="utf-8",
)
summary_scientific

## Critério final de adequação

O banco pode sustentar o treinamento quando:

1. a diversidade permanece após considerar a periodicidade;
2. a qualidade energética é controlada;
3. redundâncias físicas são identificadas;
4. multimodalidade é tratada por cluster, múltiplos candidatos ou alvo canônico;
5. o Transformer usa seno–cosseno ou perda periódica;
6. treino, validação e teste são separados por problema/Hamiltoniano, e não por linhas aleatórias do mesmo problema.

Conclusão-modelo:

> Os vetores θ apresentam diversidade angular real e dimensão efetiva mensurável. As soluções foram separadas por qualidade energética e classes de equivalência observável. Como a relação problema–parâmetros é multimodal, o treinamento utiliza representação seno–cosseno e alvos canônicos por cluster, evitando médias lineares de ângulos fisicamente incompatíveis.